## Crop type mapping workflow 

### General
This Notebook runs the crop type classification workflow documented in from [Akinyemi, Rufin, et al. 2026](https://doi.org/10.1016/j.srs.2026.100416)

### Requirements
Field samples for classification are available via [Zenodo repository](https://doi.org/10.5281/zenodo.10892509). This Notebook requires a Google Earth Engine (GEE) account with sufficient processing credits and a Google Cloud Project ID for initializing the ee Python module. Helper functions for feature computation and processing in GEE come from eepypr, further dependencies are listed below.

### Notes
Parallel computation of intermediate products (STM) and export as assets for improved efficiency. Cells ending with task.start() trigger computations in GEE, for monitoring runtime and success please visit GEE Task Manager. Final map can be retrieved from cloud storage after computation is complete.

### Import & initialize

In [2]:
import os
import sys

sys.path.append(os.path.abspath('../../'))

from src import sen, sar, stm, vec

In [ ]:
import ee
ee.Initialize(project='projectname')

import time
import datetime

import json
import geopandas as gpd
import pandas as pd
import numpy as np

from osgeo import gdal
from sklearn.model_selection import KFold

### ROI

In [ ]:
# define roi boundaries as rectangle
roi = ee.Geometry.Rectangle([[3.2, 8.0],[3.4, 8.2]])

### S1 Features

In [ ]:

startDate = datetime.datetime(2021, 10, 1)
endDate = datetime.datetime(2023, 3, 31)

# where to store assets
asset_folder = 'users/username/s1_ard/' # must exist

tempbins = [[datetime.datetime(2021, 10, 1), datetime.datetime(2021, 10, 31), 's1_ard_2021-10'], 
            [datetime.datetime(2021, 11, 1), datetime.datetime(2021, 11, 30), 's1_ard_2021-11'],
            [datetime.datetime(2021, 12, 1), datetime.datetime(2021, 12, 31), 's1_ard_2021-12'],
            [datetime.datetime(2022,  1, 1), datetime.datetime(2022,  1, 31), 's1_ard_2022-01'],
            [datetime.datetime(2022,  2, 1), datetime.datetime(2022,  2, 28), 's1_ard_2022-02'],
            [datetime.datetime(2022,  3, 1), datetime.datetime(2022,  3, 31), 's1_ard_2022-03'],
            [datetime.datetime(2022,  4, 1), datetime.datetime(2022,  4, 30), 's1_ard_2022-04'],
            [datetime.datetime(2022,  5, 1), datetime.datetime(2022,  5, 31), 's1_ard_2022-05'],
            [datetime.datetime(2022,  6, 1), datetime.datetime(2022,  6, 30), 's1_ard_2022-06'],
            [datetime.datetime(2022,  7, 1), datetime.datetime(2022,  7, 31), 's1_ard_2022-07'],
            [datetime.datetime(2022,  8, 1), datetime.datetime(2022,  8, 31), 's1_ard_2022-08'],
            [datetime.datetime(2022,  9, 1), datetime.datetime(2022,  9, 30), 's1_ard_2022-09'],
            [datetime.datetime(2022, 10, 1), datetime.datetime(2022, 10, 31), 's1_ard_2022-10'],
            [datetime.datetime(2022, 11, 1), datetime.datetime(2022, 11, 30), 's1_ard_2022-11'],
            [datetime.datetime(2022, 12, 1), datetime.datetime(2022, 12, 31), 's1_ard_2022-12'],
            [datetime.datetime(2023,  1, 1), datetime.datetime(2023,  1, 31), 's1_ard_2023-01'],
            [datetime.datetime(2023,  2, 1), datetime.datetime(2023,  2, 28), 's1_ard_2023-02'],
            [datetime.datetime(2023,  3, 1), datetime.datetime(2023,  3, 31), 's1_ard_2023-03']]

In [ ]:
# config for s1 collection
collection = sar.s1_grd_asc(startDate, endDate)

# process features for temp bins
for s, [startDate, endDate, ref_img] in enumerate(tempbins, start=1):
    print('\nCreating S1 reference for bin ' + str(s))
    print('Starting ' + str(startDate) + ', ending ' + str(endDate))

    # filter collection
    ref = collection.filterDate(startDate, endDate).select(['VHg0', 'VVg0', 'Ratio'])
    # reduce mean
    img = ref.reduce(ee.Reducer.mean()).rename(['vh_avg', 'vv_avg', 'cr_avg'])

    # export as asset
    print('Storing reference as ' + ref_img)
    task = ee.batch.Export.image.toAsset(**{
        'image': img,
        'scale': 10,
        'region': roi,
        'description': ref_img,
        'assetId': asset_folder + ref_img,
        'maxPixels': 1e13
    })
    
    task.start()

### S2 features

In [ ]:
# define region of interest
roi = ee.Geometry.Rectangle([[3.2, 8.0],[3.4, 8.2]])

# where to store assets
asset_folder = 'users/username/s2_ard/' # must exist

In [ ]:
# define time range for temporal bins and corresponding asset names
tempbins = [[datetime.datetime(2021, 10, 1), datetime.datetime(2021, 11, 30), 's2_ard_2021-10-11'],
            [datetime.datetime(2021, 12, 1), datetime.datetime(2022,  1, 31), 's2_ard_2021-12-01'],
            [datetime.datetime(2022,  2, 1), datetime.datetime(2022,  3, 31), 's2_ard_2022-02-03'],
            [datetime.datetime(2022, 10, 1), datetime.datetime(2022, 11, 30), 's2_ard_2022-10-11'],
            [datetime.datetime(2022, 12, 1), datetime.datetime(2023,  1, 31), 's2_ard_2022-12-01'],
            [datetime.datetime(2023,  2, 1), datetime.datetime(2023,  3, 31), 's2_ard_2023-02-03']]

In [ ]:
# process features for temp bins
for s, [startDate, endDate, ref_img] in enumerate(tempbins, start=1):
    print('\nCreating S2 reference for bin ' + str(s))
    print('Starting ' + str(startDate) + ', ending ' + str(endDate))

    img = stm.SEN_STM(startDate, endDate,
                          bands=['blue', 'green', 'red', 'rededge1', 'rededge2', 'rededge3', 'nir', 'swir1', 'swir2', 'ndvi', 'ndmi'],
                          metrics=['median', 'sd', 'p25', 'p75', 'iqr', 'imean']).toInt16()

    # export as asset
    print('Storing reference as ' + ref_img)
    task = ee.batch.Export.image.toAsset(**{
        'image': img,
        'scale': 10,
        'region': roi,
        'description': ref_img,
        'assetId': asset_folder + ref_img,
        'maxPixels': 1e13
    })
    
    task.start()

### Prepare classification

In [ ]:
# local shapefile with training points
point_shape = '/training.gpkg'

# class name column
target = 'class_id'

# read training vector
points = gpd.read_file(point_shape)

In [ ]:
# crs manipulation
points.crs
if False:
    points = points.to_crs(epsg=4326)

In [ ]:
# print overview of class occurrence
pd.unique(points[target])
points[target].value_counts()
points[target].value_counts()

for col in points.columns:
    print(col)
points.head()

# remove other attributes
points = points.drop(['id', 'path'], axis=1)
points = points[points[target].notna()]


In [ ]:
# create geojson from geopandas
point_f = []
for i in range(points.shape[0]):
    geom = points.iloc[i:i + 1, :]
    jsonDict = eval(geom.to_json())
    geojsonDict = jsonDict['features'][0]
    point_f.append(ee.Feature(geojsonDict))

# make feature collection
ptsfc = ee.FeatureCollection(point_f)

In [ ]:
# stack s1 and s2 assets

feat = ['users/philipperufin/s2_ard/s2_ard_2021-10-11',
        'users/philipperufin/s2_ard/s2_ard_2021-12-01',
        'users/philipperufin/s2_ard/s2_ard_2022-02-03',
        'users/philipperufin/s2_ard/s2_ard_2022-10-11',
        'users/philipperufin/s2_ard/s2_ard_2022-12-01',
        'users/philipperufin/s2_ard/s2_ard_2023-02-03',

        'users/philipperufin/s1_ard/s1_ard_2021-10',
        'users/philipperufin/s1_ard/s1_ard_2021-11',
        'users/philipperufin/s1_ard/s1_ard_2021-12',
        'users/philipperufin/s1_ard/s1_ard_2022-01',
        'users/philipperufin/s1_ard/s1_ard_2022-02',
        'users/philipperufin/s1_ard/s1_ard_2022-03',
        'users/philipperufin/s1_ard/s1_ard_2022-04',
        'users/philipperufin/s1_ard/s1_ard_2022-05',
        'users/philipperufin/s1_ard/s1_ard_2022-06',
        'users/philipperufin/s1_ard/s1_ard_2022-07',
        'users/philipperufin/s1_ard/s1_ard_2022-08',
        'users/philipperufin/s1_ard/s1_ard_2022-09',
        'users/philipperufin/s1_ard/s1_ard_2022-10',
        'users/philipperufin/s1_ard/s1_ard_2022-11',
        'users/philipperufin/s1_ard/s1_ard_2022-12',
        'users/philipperufin/s1_ard/s1_ard_2023-01',
        'users/philipperufin/s1_ard/s1_ard_2023-02',
        'users/philipperufin/s1_ard/s1_ard_2023-03']

stm_image = ee.Image([feat])
bands = stm_image.bandNames().getInfo()
len(bands)


In [ ]:
# sample point locations
stm = stm_image.sampleRegions(ptsfc, [target], 10)

### Train RF model

In [ ]:
# classifier
# define band names
bands = stm_image.bandNames().getInfo()
len(bands)

classifier = ee.Classifier.smileRandomForest(250).train(stm, target, bands, subsamplingSeed=42042)
multiprob = ee.Classifier.smileRandomForest(250).setOutputMode('MULTIPROBABILITY').train(stm, target, bands, subsamplingSeed=42042)


In [ ]:
# obtain out-of-bag (OOB), confusion matrix, and variable importance
oob = classifier.confusionMatrix().accuracy().getInfo()
con = classifier.confusionMatrix().getInfo()
var = classifier.explain().get('importance').getInfo()

In [ ]:
# export variable importance as table - accounts for various dictionary structures in GEE var imp objects
out_file_var_imp = '/target_directory/variable_importance.csv'

def export_importance(importance_dict, predictors=None):
    processed_data = []

    for variable, importance_value in importance_dict.items():
        # handle different data types
        if isinstance(importance_value, (int, float)):
            # single numeric value
            processed_data.append({'Variable': variable, 'Importance': importance_value})
        elif isinstance(importance_value, list):
            # list of values - take the first one or average
            if importance_value:
                if all(isinstance(x, (int, float)) for x in importance_value):
                    # use average if multiple numeric values
                    processed_data.append({
                        'Variable': variable,
                        'Importance': np.mean(importance_value)
                    })
                else:
                    # take first element if numeric
                    first_val = importance_value[0]
                    if isinstance(first_val, (int, float)):
                        processed_data.append({
                            'Variable': variable,
                            'Importance': first_val
                        })
            else:
                processed_data.append({'Variable': variable, 'Importance': 0.0})
        
        elif isinstance(importance_value, dict):
            # nested dictionary - extract numeric values
            numeric_values = [v for v in importance_value.values() if isinstance(v, (int, float))]
            if numeric_values:
                processed_data.append({
                    'Variable': variable,
                    'Importance': np.mean(numeric_values)
                })
        else:
            # unknown type, default to 0
            print(f"Warning: Unknown type for {variable}: {type(importance_value)}")
            processed_data.append({'Variable': variable, 'Importance': 0.0})

    # create DataFrame
    df = pd.DataFrame(processed_data)

    if predictors is not None:
        existing_vars = set(df['Variable'])
        missing_vars = [p for p in predictors if p not in existing_vars]

        for var in missing_vars:
            df = pd.concat([df, pd.DataFrame([{
                'Variable': var,
                'Importance': 0.0
            }])], ignore_index=True)

    # sort by importance
    df = df.sort_values('Importance', ascending=False).reset_index(drop=True)

    # calculate additional metrics
    total_importance = df['Importance'].sum()
    if total_importance > 0:
        df['Importance_Percentage'] = (df['Importance'] / total_importance) * 100
        df['Cumulative_Percentage'] = df['Importance_Percentage'].cumsum()
        df['Normalized_Importance'] = df['Importance'] / df['Importance'].max()
    else:
        df['Importance_Percentage'] = 0.0
        df['Cumulative_Percentage'] = 0.0
        df['Normalized_Importance'] = 0.0

    df['Rank'] = df.index + 1

    return df

df_importance = export_importance(var, predictors=bands)
df_importance.to_csv(out_file_var_imp, index=False, sep=';', header=True)

### Predict

In [ ]:
# define roi boundaries
roi_path = '/roi.gpkg'
roi_vect = gpd.read_file(roi_path)

g = json.loads(roi_vect.to_json())
coords = list(g['features'][0]['geometry']['coordinates'])
roi_poly = ee.Geometry.Polygon(coords)

In [ ]:
### create classification
map = stm_image.classify(classifier).toInt8()

In [ ]:
# export as asset
task = ee.batch.Export.image.toAsset(**{
    'image': map,
    'scale': 10,
    'region': roi_poly,
    'description': 'crops',
    'assetId': 'crops',
    'maxPixels': 1e13
})

task.start()

In [ ]:
# create multiprobability outputs
prb = stm_image.classify(multiprob).arrayFlatten(coordinateLabels=[['cassava-ep', 'cassava-lp', 'maize-cassava', 'maize-ep', 'maize-lp', 'rice', 'yam', 'others']]).multiply(10000).toInt16()

# export as asset
task = ee.batch.Export.image.toAsset(**{
    'image': prb,
    'scale': 10,
    'region': roi_poly,
    'description': 'crops_probs',
    'assetId': 'crops_probs',
    'maxPixels': 1e13
})

task.start()

### Calculate probability margins & binary active learning layer

In [ ]:
# expects class probability raster f on disk
f = '/map_dir/crops_probs.tif'
n_classes = 8

# open
ds = gdal.Open(f)
ar = ds.ReadAsArray()

# calculate probability margins
sr = np.sort(ar, axis=0)
mr = sr[n_classes-1,:,:] - sr[n_classes-2,:,:]

# create raster in memory
drvMemR = gdal.GetDriverByName('MEM')
mem_ds = drvMemR.Create('', ds.RasterXSize, ds.RasterYSize, 1, gdal.GDT_Int16)
mem_ds.SetGeoTransform(ds.GetGeoTransform())
mem_ds.SetProjection(ds.GetProjection())
driver = gdal.GetDriverByName("GTiff")
mem_ds.GetRasterBand(1).WriteArray(mr)

# store to disk
copy_ds = driver.CreateCopy(f[:-4]+'_margin.tif', mem_ds, 0, options=["COMPRESS=LZW"])
copy_ds = None

In [ ]:
# read probability margins
print('read margins')
marg = '/map_dir/crops_probs_margin.tif'
marg_ar = gdal.Open(marg).ReadAsArray()

# read crop type map
print('read crop type map')
crop = '/map_dir/crops.tif'
crop_ar = gdal.Open(crop).ReadAsArray()

# read LC mask - cropland mask e.g. obtained from worldcover
print('read lc mask')
mask = '/map_dir/worldcover_mask.tif'
mask_ar = gdal.Open(mask).ReadAsArray()

# mask crop type map
crop_ar[mask_ar==0] = 0

In [ ]:
# iterate over n classes, calculate 25th percentile of margins
for c in range(1, n_classes+1):
    print(c)
    # subset probability margin values of class c
    sub = marg_ar[(crop_ar==c)]
    # calculate 25th percentile for each class
    p25 = np.nanpercentile(sub, 25)
    print(p25)
    # set lc values at higher margins to 0
    crop_ar[(crop_ar==c) & (marg_ar>p25)] = 0

print('writing active learning mask (uncertain lc predictions)')
drvMemR = gdal.GetDriverByName('MEM')
mem_ds = drvMemR.Create('', gdal.Open(crop).RasterXSize, gdal.Open(crop).RasterYSize, 1, gdal.GDT_Byte)
mem_ds.SetGeoTransform(gdal.Open(crop).GetGeoTransform())
mem_ds.SetProjection(gdal.Open(crop).GetProjection())

results = mem_ds.GetRasterBand(1)
results.WriteArray(crop_ar)

creationOptions = ['COMPRESS=LZW']
driver = gdal.GetDriverByName("GTiff")
copy_ds = driver.CreateCopy('/map_dir/crops_probs_margin_25prc.tif', mem_ds, 0, options=creationOptions)
copy_ds = None

### k-fold CV

In [ ]:
point_shape = '/training.gpkg'
out_path = '/validation/'
fold = 30

### read training vector
points = gpd.read_file(point_shape)


### create k-fold splits
kf = KFold(n_splits=fold, random_state=42, shuffle=True)
kf.get_n_splits(points)

for subset, (train_index, test_index) in enumerate(kf.split(points)):

    print(len(train_index))
    print(len(test_index))

    train_points = points.iloc[train_index]
    test_points = points.iloc[test_index]

    ### class distribution
    print(train_points[target].value_counts())

    # train data
    point_f = []
    for i in range(train_points.shape[0]):
        geom = train_points.iloc[i:i + 1, :]
        jsonDict = eval(geom.to_json())
        geojsonDict = jsonDict['features'][0]
        point_f.append(ee.Feature(geojsonDict))

    # make feature collection
    ptsfc = ee.FeatureCollection(point_f)

    # test data
    point_t = []
    for i in range(test_points.shape[0]):
        geom = test_points.iloc[i:i + 1, :]
        jsonDict = eval(geom.to_json())
        geojsonDict = jsonDict['features'][0]
        point_t.append(ee.Feature(geojsonDict))

    # make feature collection
    ptsft = ee.FeatureCollection(point_t)
    
    # stack input features as defined in feat above
    stm_image = ee.Image([feat])
    bands = stm_image.bandNames().getInfo()
    print('n feats: ' + str(len(bands)))

    ###########################################################
    ###########################################################
    # sample point locations
    stm = stm_image.sampleRegions(ptsfc, [target], 10)

    ###########################################################
    # classifier
    # define band names
    bands = stm_image.bandNames().getInfo()

    classifier = ee.Classifier.smileRandomForest(250).train(stm, target, bands, subsamplingSeed=42042)

    oob = classifier.confusionMatrix().accuracy().getInfo()
    con = classifier.confusionMatrix().getInfo()
    var = classifier.explain().getInfo()

    ### predict & sample
    map = stm_image.classify(classifier).toInt8()
    prd = map.sampleRegions(ptsft, [target], 10)
    prd.getInfo()

    ref_l = prd.aggregate_array(target).getInfo()
    prd_l = prd.aggregate_array('classification').getInfo()

    df = pd.DataFrame(list(zip(prd_l, ref_l)), columns =['prediction', 'reference'])
    df.to_csv(out_path + f'/results_{subset:02d}.csv', index=False, sep=';', header=True)